# MAPPO Maze Exploration Curriculum

Train a writable-byte exploration policy on generated wide-corridor mazes from `10x10` through `50x50`.


In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status


In [ ]:
import importlib

import jax

from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")


## Quick Smoke Run

Run one tiny job before starting the maze curriculum.


In [ ]:
smoke_metrics = workflows.run_jax_smoke(jax_runner.main)
smoke_metrics


## Curriculum Settings

Edit `experiments/maze_exploration_curriculum.json` for durable maze, budget, or temperature changes.


In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "maze_exploration_curriculum.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)
experiment_args = dict(experiment.args)
experiment_metadata = dict(experiment.metadata or {})

STAGE_SIZES = tuple(int(size) for size in experiment_metadata["stage_sizes"])
CURRICULUM_STAGES = workflows.build_maze_exploration_curriculum_stages(
    STAGE_SIZES,
    training_profile=experiment_metadata.get("stage_training_profile"),
)
RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "maze_exploration_curriculum"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
ROLLOUT_POLICY_TEMPERATURE = workflows.notebook_rollout_policy_temperature(experiment_metadata)
WANDB_VIDEO_MAX_FRAMES = experiment_metadata.get("wandb_video_max_frames")
WANDB_VIDEO_STAGE_NAMES = experiment_metadata.get("wandb_video_stage_names")
GLOBAL_UPDATE_CAP = int(experiment_metadata["global_update_cap"])
UPDATE_TIMESTEPS = workflows.update_timesteps(
    num_envs=int(experiment_args["num_envs"]),
    num_steps=int(experiment_args["num_steps"]),
)

COMMON_ARGS = workflows.build_maze_exploration_common_args(
    CURRICULUM_STAGES,
    num_envs=int(experiment_args["num_envs"]),
    num_steps=int(experiment_args["num_steps"]),
    actor_vision_radius=int(experiment_args["actor_vision_radius"]),
    write_bits=int(experiment_args["write_bits"]),
    gamma=float(experiment_args["gamma"]),
    seed=int(experiment_args["seed"]),
    maze_corridor_width=int(experiment_args["maze_corridor_width"]),
    maze_wall_width=int(experiment_args["maze_wall_width"]),
    maze_seed=int(experiment_args["maze_seed"]),
)
COMMON_ARGS += ["--exp-name", str(experiment_args["exp_name"])]

WANDB_PROJECT = "cool-antz"
WANDB_ENTITY = None
WANDB_GROUP = "maze_exploration_curriculum"
WANDB_MODE = "online"

{
    "stage_range": experiment_metadata["stage_range"],
    "stage_count": len(CURRICULUM_STAGES),
    "global_update_cap": GLOBAL_UPDATE_CAP,
    "update_timesteps": UPDATE_TIMESTEPS,
    "maze_corridor_width": experiment_args["maze_corridor_width"],
    "maze_wall_width": experiment_args["maze_wall_width"],
    "write_bits": experiment_args["write_bits"],
    "rollout_policy_temperature": ROLLOUT_POLICY_TEMPERATURE,
    "wandb_video_stage_names": WANDB_VIDEO_STAGE_NAMES,
}


## Train Maze Curriculum


In [ ]:
maze_result = workflows.run_forage_curriculum(
    stages=CURRICULUM_STAGES,
    checkpoint_dir=CHECKPOINT_DIR,
    common_args=COMMON_ARGS,
    update_timesteps_per_stage=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_group=WANDB_GROUP,
    wandb_run_name="maze_exploration_curriculum",
    wandb_mode=WANDB_MODE,
    wandb_tags=["mappo", "exploration", "maze"],
    wandb_notes=experiment_metadata.get("notes"),
    wandb_artifact_paths=[EXPERIMENT_CONFIG],
    wandb_artifact_prefix="maze-exploration-curriculum",
    checkpoint_name_prefix="jax_mappo_maze_explore",
    wandb_video_key_prefix="videos/maze_exploration",
    wandb_video_max_frames=WANDB_VIDEO_MAX_FRAMES,
    wandb_video_stage_names=WANDB_VIDEO_STAGE_NAMES,
    wandb_video_policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
)
FINAL_CHECKPOINT = maze_result["final_checkpoint_path"]
FINAL_CHECKPOINT
